# 01 – Data Exploration

Load StatsBomb open data for Euro 2020, World Cup 2022 and Euro 2024.
Inspect available competitions, matches, event types and 360 frame coverage.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
from statsbombpy import sb

from src.config import COMPETITIONS


## Available competitions with 360 data


In [ ]:
comps = sb.competitions()
# Show competitions we care about
for c in COMPETITIONS:
    row = comps[
        (comps['competition_id'] == c['competition_id']) &
        (comps['season_id'] == c['season_id'])
    ]
    if row.empty:
        print(f"WARNING: {c['name']} (competition_id={c['competition_id']}, season_id={c['season_id']}) NOT found.")
    else:
        print(f"OK: {c['name']} – {row[['competition_name','season_name']].to_string(index=False)}")


## Load one competition (quick sample)


In [ ]:
from src.data.loader import load_competition

# Start with Euro 2020 (has full 360 coverage)
events, frames, lineups, matches = load_competition(
    competition_id=55, season_id=43
)
print(f'Matches: {len(matches)}')
print(f'Events: {len(events):,}')
print(f'Frame rows: {len(frames):,}')


## Event type distribution


In [ ]:
print(events['type'].value_counts().head(20))


## 360 frame coverage per match


In [ ]:
if not frames.empty:
    coverage = frames.groupby('match_id')['event_id'].nunique().rename('events_with_360')
    total = events.groupby('match_id')['id'].nunique().rename('total_events')
    cov_df = pd.concat([coverage, total], axis=1).fillna(0)
    cov_df['coverage_pct'] = (cov_df['events_with_360'] / cov_df['total_events'] * 100).round(1)
    display(cov_df.describe())


## Inspect a single freeze frame


In [ ]:
if not frames.empty:
    sample_event = frames['event_id'].iloc[0]
    sample_frame = frames[frames['event_id'] == sample_event]
    print(f'Event {sample_event}: {len(sample_frame)} players in frame')
    display(sample_frame[['player_name','position_name','teammate','frame_x','frame_y']].head(20))
